## Data Intake and Pre-processing

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
pd.set_option("display.max_columns", 100)

SNAPSHOT_DATE = pd.Timestamp("2026-06-30") # assumed day of analysis
TARGET_END = pd.Timestamp("2026-07-31")

In [ ]:
# Works whether Jupyter starts from repo root or project folder
ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"

In [ ]:
customers = pd.read_csv("customers.csv")
transactions = pd.read_csv(
    "transactions.csv",
    parse_dates=["order_date"]
)

In [ ]:
print(customers.shape)
print(transactions.shape)

(12000, 7)
(57856, 8)


In [ ]:
display(customers.head())
display(transactions.head())

,customer_id,age,region,acquisition_channel,tenure_months,loyalty_member,email_engagement_score
0,1,40,West,Paid Social,27,1,0.3692
1,2,46,South,Paid Search,22,0,0.4690
2,3,33,West,Organic,4,0,0.4790
3,4,49,Midwest,Partner,32,1,0.6068
4,5,52,Northeast,Organic,13,0,0.3126


,order_id,customer_id,order_date,channel,category,units,discount_pct,revenue
0,264,60,2025-01-01,Web,Home,1,0.0,137.89
1,781,170,2025-01-01,Web,Grocery,1,0.0,46.43
2,1108,237,2025-01-01,Web,Grocery,2,0.0,134.22
3,1709,359,2025-01-01,Web,Grocery,1,0.0,47.89
4,1999,405,2025-01-01,App,Grocery,2,0.0,96.51


In [ ]:
history = transactions[
    transactions["order_date"] <= SNAPSHOT_DATE
].copy()

july = transactions[
    (transactions["order_date"] > SNAPSHOT_DATE)
    & (transactions["order_date"] <= TARGET_END)
].copy()

In [ ]:
print(
    "Historical range:",
    history["order_date"].min(),
    "to",
    history["order_date"].max()
)

Historical range: 2025-01-01 00:00:00 to 2026-06-30 00:00:00


In [ ]:
print(
    "Target range:",
    july["order_date"].min(),
    "to",
    july["order_date"].max()
)

Target range: 2026-07-01 00:00:00 to 2026-07-31 00:00:00


In [ ]:
# behavioral history helper function
def build_window_features(transactions, snapshot_date, days):
    start_date = snapshot_date - pd.Timedelta(days=days - 1)

    window = transactions[
        (transactions["order_date"] >= start_date)
        & (transactions["order_date"] <= snapshot_date)
    ].copy()

    features = (
        window
        .groupby("customer_id")
        .agg(
            orders=("order_id", "nunique"),
            spend=("revenue", "sum"),
            units=("units", "sum"),
            avg_discount=("discount_pct", "mean"),
            distinct_categories=("category", "nunique"),
        )
    )

    features["avg_order_value"] = (
        features["spend"] / features["orders"]
    )

    app_share = (
        window
        .assign(is_app=(window["channel"] == "App").astype(int))
        .groupby("customer_id")["is_app"]
        .mean()
    )

    features["app_order_share"] = app_share

    features = features.add_suffix(f"_{days}d")

    return features.reset_index()

In [ ]:
features_30d = build_window_features(
    history,
    SNAPSHOT_DATE,
    30
)

features_90d = build_window_features(
    history,
    SNAPSHOT_DATE,
    90
)

features_180d = build_window_features(
    history,
    SNAPSHOT_DATE,
    180
)

In [ ]:
last_purchase = (
    history
    .groupby("customer_id")["order_date"]
    .max()
    .rename("last_order_date")
    .reset_index()
)

In [ ]:
model_df = customers.copy()

for feature_table in [
    features_30d,
    features_90d,
    features_180d,
    last_purchase
]:
    model_df = model_df.merge(
        feature_table,
        on="customer_id",
        how="left"
    )

In [ ]:
model_df["has_prior_purchase"] = (
    model_df["last_order_date"].notna()
).astype(int)

model_df["days_since_last_order"] = (
    SNAPSHOT_DATE - model_df["last_order_date"]
).dt.days

In [ ]:
model_df.shape

(12000, 31)

In [ ]:
# set record for customers who never purchased
max_possible_history = (
    SNAPSHOT_DATE - transactions["order_date"].min()
).days + 1

model_df["days_since_last_order"] = (
    model_df["days_since_last_order"]
    .fillna(max_possible_history)
)

In [ ]:
# build and merge in july outcome variable
july_buyers = (
    july
    .groupby("customer_id")
    .size()
    .gt(0)
    .rename("purchased_july")
    .astype(int)
    .reset_index()
)

model_df = model_df.merge(
    july_buyers,
    on="customer_id",
    how="left"
)

model_df["purchased_july"] = (
    model_df["purchased_july"]
    .fillna(0)
    .astype(int)
)

In [ ]:
model_df.columns

Index(['customer_id', 'age', 'region', 'acquisition_channel', 'tenure_months',
       'loyalty_member', 'email_engagement_score', 'orders_30d', 'spend_30d',
       'units_30d', 'avg_discount_30d', 'distinct_categories_30d',
       'avg_order_value_30d', 'app_order_share_30d', 'orders_90d', 'spend_90d',
       'units_90d', 'avg_discount_90d', 'distinct_categories_90d',
       'avg_order_value_90d', 'app_order_share_90d', 'orders_180d',
       'spend_180d', 'units_180d', 'avg_discount_180d',
       'distinct_categories_180d', 'avg_order_value_180d',
       'app_order_share_180d', 'last_order_date', 'has_prior_purchase',
       'days_since_last_order'],
      dtype='object')

In [ ]:
# identify and grab the behavioral columns
behavior_columns = [
    c for c in model_df.columns
    if any(
        token in c
        for token in [
            "orders_",
            "spend_",
            "units_",
            "avg_discount_",
            "distinct_categories_",
            "avg_order_value_",
            "app_order_share_",
        ]
    )
]

model_df[behavior_columns] = (
    model_df[behavior_columns]
    .fillna(0)
)

In [ ]:
# sanity checks
print("Rows:", len(model_df))
print("Unique customers:", model_df["customer_id"].nunique())

print("\nTarget distribution:")
print(
    model_df["purchased_july"]
    .value_counts()
)

print("\nTarget rate:")
print(
    model_df["purchased_july"]
    .mean()
)

print("\nMissing values:")
print(
    model_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

display(model_df.sample(5, random_state=42))

Rows: 12000
Unique customers: 12000

Target distribution:
purchased_july
0    9412
1    2588
Name: count, dtype: int64

Target rate:
0.21566666666666667

Missing values:
last_order_date            543
customer_id                  0
region                       0
age                          0
tenure_months                0
loyalty_member               0
email_engagement_score       0
acquisition_channel          0
spend_30d                    0
units_30d                    0
avg_discount_30d             0
distinct_categories_30d      0
avg_order_value_30d          0
app_order_share_30d          0
orders_90d                   0
dtype: int64


,customer_id,age,region,acquisition_channel,tenure_months,loyalty_member,email_engagement_score,orders_30d,spend_30d,units_30d,avg_discount_30d,distinct_categories_30d,avg_order_value_30d,app_order_share_30d,orders_90d,spend_90d,units_90d,avg_discount_90d,distinct_categories_90d,avg_order_value_90d,app_order_share_90d,orders_180d,spend_180d,units_180d,avg_discount_180d,distinct_categories_180d,avg_order_value_180d,app_order_share_180d,last_order_date,has_prior_purchase,days_since_last_order,purchased_july
1935,1936,31,Northeast,Paid Search,33,1,0.2202,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.000,0.0,1.0,596.81,3.0,0.0,1.0,596.8100,0.00,2026-02-14,1,136.0,0
6494,6495,45,West,Paid Search,49,1,0.2239,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.0,0.0000,0.00,2025-04-01,1,455.0,0
1720,1721,35,Northeast,Paid Social,25,0,0.3168,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,46.25,2.0,20.0,1.0,46.250,0.0,1.0,46.25,2.0,20.0,1.0,46.2500,0.00,2026-04-23,1,68.0,0
9120,9121,55,Midwest,Organic,18,1,0.2866,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,896.43,5.0,0.0,2.0,448.215,0.0,4.0,1145.85,9.0,0.0,3.0,286.4625,0.25,2026-05-05,1,56.0,0
360,361,42,West,Organic,14,0,0.7193,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.000,0.0,1.0,186.15,3.0,0.0,1.0,186.1500,1.00,2026-01-27,1,154.0,0


In [ ]:
assert len(model_df) == model_df["customer_id"].nunique()
assert history["order_date"].max() <= SNAPSHOT_DATE
assert july["order_date"].min() > SNAPSHOT_DATE
assert model_df["purchased_july"].isin([0, 1]).all()

## Preliminary Data Analysis

In [ ]:
numeric_summary = (
    model_df
    .groupby("purchased_july")
    [
        [
            "email_engagement_score",
            "tenure_months",
            "orders_30d",
            "spend_30d",
            "orders_90d",
            "spend_90d",
            "orders_180d",
            "spend_180d",
            "days_since_last_order"
        ]
    ]
    .mean()
    .T
)

numeric_summary.columns = [
    "No July Purchase",
    "July Purchase"
]

numeric_summary["difference"] = (
    numeric_summary["July Purchase"]
    - numeric_summary["No July Purchase"]
)

display(numeric_summary)

,No July Purchase,July Purchase,difference
email_engagement_score,0.458724,0.503379,0.044655
tenure_months,23.732363,28.253864,4.521501
orders_30d,0.228007,0.319165,0.091159
spend_30d,44.924569,63.418056,18.493488
orders_90d,0.706439,0.937017,0.230578
spend_90d,137.598879,186.187342,48.588462
orders_180d,1.406609,1.910742,0.504133
spend_180d,274.203231,388.982117,114.778886
days_since_last_order,149.864960,116.375193,-33.489766


In [ ]:
for col in [
    "region",
    "acquisition_channel",
    "loyalty_member"
]:
    print(f"\n### {col}")

    display(
        pd.crosstab(
            model_df[col],
            model_df["purchased_july"],
            normalize="index"
        ).rename(
            columns={
                0: "No Purchase",
                1: "Purchase"
            }
        )
    )


### region


purchased_july,No Purchase,Purchase
region,,
Midwest,0.790282,0.209718
Northeast,0.788484,0.211516
South,0.779347,0.220653
West,0.783088,0.216912



### acquisition_channel


purchased_july,No Purchase,Purchase
acquisition_channel,,
Organic,0.793069,0.206931
Paid Search,0.785664,0.214336
Paid Social,0.766625,0.233375
Partner,0.792621,0.207379
Referral,0.784393,0.215607



### loyalty_member


purchased_july,No Purchase,Purchase
loyalty_member,,
0,0.846736,0.153264
1,0.730710,0.269290


## Preparation for Modeling

In [ ]:
target = "purchased_july"

categorical_features = [
    "region",
    "acquisition_channel"
]

numeric_features = [
    "age",
    "tenure_months",
    "loyalty_member",
    "email_engagement_score",
    "has_prior_purchase",
    "days_since_last_order",

    "orders_30d",
    "spend_30d",
    "units_30d",
    "avg_discount_30d",
    "distinct_categories_30d",
    "avg_order_value_30d",
    "app_order_share_30d",

    "orders_90d",
    "spend_90d",
    "units_90d",
    "avg_discount_90d",
    "distinct_categories_90d",
    "avg_order_value_90d",
    "app_order_share_90d",

    "orders_180d",
    "spend_180d",
    "units_180d",
    "avg_discount_180d",
    "distinct_categories_180d",
    "avg_order_value_180d",
    "app_order_share_180d",
]

X = model_df[
    categorical_features + numeric_features
].copy()

y = model_df[target].copy()

In [ ]:
# train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Train target rate:",
    round(y_train.mean(), 4)
)

print(
    "Test target rate:",
    round(y_test.mean(), 4)
)

Training rows: 9600
Test rows: 2400
Train target rate: 0.2156
Test target rate: 0.2158


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

In [ ]:
# numeric features pipeline
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [ ]:
# categorical features pipeline
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

### Logistic regression baseline

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]
)

In [ ]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv_results = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

In [ ]:
cv_summary = pd.DataFrame({
    metric: [
        cv_results[f"train_{metric}"].mean(),
        cv_results[f"test_{metric}"].mean()
    ]
    for metric in scoring
}).T

cv_summary.columns = [
    "Train",
    "Validation"
]

display(cv_summary.round(3))

,Train,Validation
roc_auc,0.639,0.624
pr_auc,0.328,0.311
precision,0.566,0.511
recall,0.018,0.016
f1,0.036,0.031


### Holdout evaluation

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

logistic_model.fit(
    X_train,
    y_train
)

test_prob = logistic_model.predict_proba(
    X_test
)[:, 1]

test_pred = logistic_model.predict(
    X_test
)

metrics = {
    "ROC-AUC": roc_auc_score(
        y_test,
        test_prob
    ),
    "PR-AUC": average_precision_score(
        y_test,
        test_prob
    ),
    "Precision": precision_score(
        y_test,
        test_pred,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        test_pred
    ),
    "F1": f1_score(
        y_test,
        test_pred
    )
}

pd.Series(metrics).round(3)

,0
ROC-AUC,0.647
PR-AUC,0.331
Precision,0.571
Recall,0.015
F1,0.030


### Decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=5,
                min_samples_leaf=50,
                random_state=42
            )
        )
    ]
)

###Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=20,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": forest_model
}

comparison = []

for model_name, model in models.items():

    result = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True
    )

    comparison.append({
        "Model": model_name,

        "Train ROC-AUC":
            result["train_roc_auc"].mean(),

        "CV ROC-AUC":
            result["test_roc_auc"].mean(),

        "CV PR-AUC":
            result["test_pr_auc"].mean(),

        "CV Precision":
            result["test_precision"].mean(),

        "CV Recall":
            result["test_recall"].mean(),

        "CV F1":
            result["test_f1"].mean(),
    })

model_comparison = pd.DataFrame(
    comparison
).set_index("Model")

display(model_comparison.round(3))

,Train ROC-AUC,CV ROC-AUC,CV PR-AUC,CV Precision,CV Recall,CV F1
Model,,,,,,
Logistic Regression,0.639,0.624,0.311,0.511,0.016,0.031
Decision Tree,0.661,0.603,0.283,0.428,0.015,0.030
Random Forest,0.828,0.616,0.311,0.700,0.004,0.009


In [ ]:
for model_name, model in models.items():
    model.fit(X_train, y_train)

    prob = model.predict_proba(X_test)[:, 1]

    print(f"\n{model_name}")
    print(pd.Series(prob).describe(
        percentiles=[.5, .75, .9, .95, .99]
    ))
    print("Fraction >= 0.50:", (prob >= .50).mean())


Logistic Regression
count    2400.000000
mean        0.218230
std         0.085787
min         0.073337
50%         0.204653
75%         0.267857
90%         0.331989
95%         0.379235
99%         0.482261
max         0.633169
dtype: float64
Fraction >= 0.50: 0.005833333333333334

Decision Tree
count    2400.000000
mean        0.218490
std         0.096942
min         0.014493
50%         0.189843
75%         0.301119
90%         0.344609
95%         0.361991
99%         0.446541
max         0.593750
dtype: float64
Fraction >= 0.50: 0.005416666666666667

Random Forest
count    2400.000000
mean        0.218163
std         0.077270
min         0.067003
50%         0.209488
75%         0.270935
90%         0.322914
95%         0.356592
99%         0.426602
max         0.590888
dtype: float64
Fraction >= 0.50: 0.00125


In [ ]:
from sklearn.metrics import confusion_matrix

forest_model.fit(X_train, y_train)

forest_prob = forest_model.predict_proba(X_test)[:, 1]
forest_pred = (forest_prob >= 0.50).astype(int)

cm = confusion_matrix(y_test, forest_pred)

print(cm)

[[1881    1]
 [ 516    2]]


### Trace decision thresholds on logistic model


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

logistic_model.fit(X_train, y_train)

prob = logistic_model.predict_proba(X_test)[:, 1]

threshold_results = []

for threshold in np.arange(0.10, 0.51, 0.05):

    pred = (prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_test, pred, zero_division=0
        ),
        "recall": recall_score(
            y_test, pred
        ),
        "f1": f1_score(
            y_test, pred
        ),
        "predicted_positive_rate": pred.mean()
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(3))

NameError: name 'logistic_model' is not defined